# Kernel SVM

**Learning goal:** Predict whether a user purchases a product (`Purchased`: 1 = Yes, 0 = No) based on **Age** and **EstimatedSalary**. The given CSV contains 400 records. We preserve the workflow of your original notebook and explain every step.

## Importing the libraries

We import NumPy for arrays, Pandas for the CSV and Matplotlib for plots.

In [ ]:
# NumPy: arrays and numerical operations
import numpy as np
# Matplotlib: plots and the SVM decision boundary
import matplotlib.pyplot as plt
# Pandas: loading and exploring the CSV dataset
import pandas as pd

## Importing the dataset

The columns are `User ID`, `Gender`, `Age`, `EstimatedSalary`, and `Purchased`. Only Age and EstimatedSalary are input features; Purchased is the target. The ID should not be treated as a predictive feature.

In [ ]:
# Read the CSV file from the same folder as this notebook.
dataset = pd.read_csv('Social_Network_Ads.csv')
# Select columns 2 and 3 (0-based indexing): Age and EstimatedSalary.
X = dataset.iloc[:, [2, 3]].values
# Select the final column: Purchased (0 = No, 1 = Yes).
y = dataset.iloc[:, -1].values
# Display the first five rows so students can see the input.
print(dataset.head())
print('Dataset rows:', len(dataset), '| Features:', X.shape[1])

## Splitting the dataset into the Training set and Test set

**Train/test split:** 75% for learning and 25% for checking performance on unseen data. `random_state=0` makes the same split every run.

In [ ]:
# train_test_split divides data into training and testing portions.
from sklearn.model_selection import train_test_split
# 25% test data; 75% training data; random_state gives repeatable results.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)
print('Training:', X_train.shape, '| Testing:', X_test.shape)

## Feature Scaling

**Why scaling?** Salary values are much larger than age. Standardizing both features prevents a large numeric range from dominating SVM distances. Fit the scaler **only on training data** to avoid data leakage.

In [ ]:
# SVM is sensitive to scale, so standardize Age and EstimatedSalary.
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
# Learn mean and standard deviation from training data, then scale it.
X_train = sc.fit_transform(X_train)
# Apply EXACTLY the same training-based transformation to test data.
X_test = sc.transform(X_test)

## Training the Kernel SVM model on the Training set

**Model training:** The SVM identifies a decision boundary separating purchasers and non-purchasers. Support vectors are the influential training examples closest to the boundary.

In [ ]:
# SVC is scikit-learn's Support Vector Classifier.
from sklearn.svm import SVC
# A linear kernel creates a straight decision boundary.
# An RBF kernel can produce a curved/nonlinear decision boundary.
classifier = SVC(kernel='rbf', random_state=0,c=0.1)
# Learn the boundary from the TRAINING set only.
classifier.fit(X_train, y_train)
# Count the training points that act as support vectors.
print('Support vectors per class:', classifier.n_support_)

## Predicting the Test set results

Use the fitted classifier to predict 0 or 1 for the held-out test rows.

In [ ]:
# Predict purchase labels for unseen test examples.
y_pred = classifier.predict(X_test)
# Show a few actual vs predicted labels (1 = Purchased).
print('Actual labels:   ', y_test[:10])
print('Predicted labels:', y_pred[:10])

## Making the Confusion Matrix

The confusion matrix shows correct vs incorrect predictions: rows = actual, columns = predicted. In binary classification, matrix entries are TN, FP, FN, TP.

In [ ]:
# Confusion matrix: rows = actual classes, columns = predicted classes.
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix (TN, FP / FN, TP):\n', cm)
# Accuracy is the proportion of correct predictions.
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
# Precision, recall and F1-score offer extra classification information.
print('Classification report:\n', classification_report(y_test, y_pred, zero_division=0))

## Visualising the Training set results

**Training-set plot:** shaded regions show predicted classes; dots show actual labels. Both axes represent **standardized** values after scaling.

In [ ]:
# Draw the decision regions and the real data points (training set).
from matplotlib.colors import ListedColormap
# The model was trained on standardized features; use standardized coordinates.
X_set, y_set = X_train, y_train
# Generate evenly spaced coordinates across the two feature axes.
# Step 0.03 keeps plotting practical on classroom computers.
X1, X2 = np.meshgrid(
    np.arange(X_set[:, 0].min() - 1, X_set[:, 0].max() + 1, 0.03),
    np.arange(X_set[:, 1].min() - 1, X_set[:, 1].max() + 1, 0.03)
)
# Predict every grid point to display the regions assigned to each class.
grid_points = np.c_[X1.ravel(), X2.ravel()]
Z = classifier.predict(grid_points).reshape(X1.shape)
plt.figure(figsize=(8, 5))
plt.contourf(X1, X2, Z, alpha=0.25, cmap=ListedColormap(('red', 'green')))
# Plot actual users: red = not purchased (0), green = purchased (1).
for label, color in [(0, 'red'), (1, 'green')]:
    plt.scatter(X_set[y_set == label, 0], X_set[y_set == label, 1],
                color=color, edgecolors='black', s=35, label=f'Purchased = {label}')
# Mark training support vectors (they define or influence the margin).
if 'train' == 'train':
    plt.scatter(classifier.support_vectors_[:, 0], classifier.support_vectors_[:, 1],
                facecolors='none', edgecolors='blue', s=100, linewidths=1.1,
                label='Support vectors')
plt.title('RBF SVM (Training set)')
plt.xlabel('Age (standardized)')
plt.ylabel('Estimated Salary (standardized)')
plt.legend()
plt.tight_layout()
plt.show()

## Visualising the Test set results

**Test-set plot:** examine how the trained boundary classifies new users. Mistakes appear when a point is in the opposite-colored region.

In [ ]:
# Draw the decision regions and the real data points (test set).
from matplotlib.colors import ListedColormap
# The model was trained on standardized features; use standardized coordinates.
X_set, y_set = X_test, y_test
# Generate evenly spaced coordinates across the two feature axes.
# Step 0.03 keeps plotting practical on classroom computers.
X1, X2 = np.meshgrid(
    np.arange(X_set[:, 0].min() - 1, X_set[:, 0].max() + 1, 0.03),
    np.arange(X_set[:, 1].min() - 1, X_set[:, 1].max() + 1, 0.03)
)
# Predict every grid point to display the regions assigned to each class.
grid_points = np.c_[X1.ravel(), X2.ravel()]
Z = classifier.predict(grid_points).reshape(X1.shape)
plt.figure(figsize=(8, 5))
plt.contourf(X1, X2, Z, alpha=0.25, cmap=ListedColormap(('red', 'green')))
# Plot actual users: red = not purchased (0), green = purchased (1).
for label, color in [(0, 'red'), (1, 'green')]:
    plt.scatter(X_set[y_set == label, 0], X_set[y_set == label, 1],
                color=color, edgecolors='black', s=35, label=f'Purchased = {label}')
# Mark training support vectors (they define or influence the margin).
if 'test' == 'train':
    plt.scatter(classifier.support_vectors_[:, 0], classifier.support_vectors_[:, 1],
                facecolors='none', edgecolors='blue', s=100, linewidths=1.1,
                label='Support vectors')
plt.title('RBF SVM (Test set)')
plt.xlabel('Age (standardized)')
plt.ylabel('Estimated Salary (standardized)')
plt.legend()
plt.tight_layout()
plt.show()

## Predict a new customer (class demonstration)
A customer is **30 years old** and earns **₹70,000**. Always apply the same scaler before prediction. Try changing age or salary and re-run the cell.

In [ ]:
# Input values must follow the training feature order: [Age, EstimatedSalary].
new_customer = np.array([[30, 70000]])
# Transform the new example using the scaler fitted on training data.
new_customer_scaled = sc.transform(new_customer)
# Predict: 0 = Not purchased, 1 = Purchased.
prediction = classifier.predict(new_customer_scaled)[0]
print('Prediction:', prediction, '-', 'Purchased' if prediction == 1 else 'Not purchased')

## Student exercises
1. Change the new customer age to 45. What happens?
2. Change `test_size=0.25` to `test_size=0.20`. Compare accuracy.
3. Try `C=0.1` and `C=10` inside `SVC(...)`. Observe the decision boundary.
4. Run both supplied notebooks. Which model gives better test accuracy?
5. Explain why `sc.transform(X_test)` is used instead of `sc.fit_transform(X_test)`.

**Key terms:** Hyperplane = decision boundary; margin = separation around boundary; support vectors = important nearby training points; kernel = mathematical method for obtaining linear or nonlinear boundaries.